# 04 - Base Fare Prediction

## Objective

The objective is to predict `base_fare` using information available for a trip.

The model is deliberately restricted to variables that are not direct components of the final fare calculation or post-trip payment information. In particular, `charge_total`, `toll_total`, and `driver_tip_payment` are excluded because they would introduce target leakage.

A representative sample of the full taxi dataset is used for model development because the raw dataset contains approximately 48.6 million records. The sampling process preserves observations across the complete April 2025 to March 2026 period.

The target is restricted to valid positive base fares. Negative fares are retained in the raw data and documented during data-quality analysis, but are excluded from supervised fare prediction because they do not represent a valid positive fare outcome.

In [3]:
# ============================================================
# 1. Load a representative sample for fare modelling
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path(r"C:\Users\arudk\Downloads\UrbanFlow_AI")
TAXI_DIR = BASE_DIR / "data" / "raw" / "taxi"

# Columns required for the fare prediction task.
#
# We intentionally exclude post-trip monetary fields such as:
# charge_total, toll_total and driver_tip_payment.
#
# The timestamp is transformed into calendar features below.
FARE_COLUMNS = [
    "provider_code",
    "pickup_timestamp",
    "rider_count",
    "distance_miles",
    "rate_class_id",
    "offline_record_flag",
    "origin_loc_id",
    "dest_loc_id",
    "base_fare"
]

# A small fraction from every monthly file gives us coverage
# across the entire competition period without loading 48.6M
# records into memory.
SAMPLE_FRACTION = 0.02
RANDOM_STATE = 42

sample_parts = []

taxi_files = sorted(TAXI_DIR.glob("*.csv"))

print("Taxi files found:", len(taxi_files))

for file_path in taxi_files:

    print(f"Sampling: {file_path.name}")

    for chunk in pd.read_csv(
        file_path,
        usecols=FARE_COLUMNS,
        chunksize=500_000
    ):

        # Sample independently within each chunk so the final
        # dataset remains manageable in memory.
        sampled = chunk.sample(
            frac=SAMPLE_FRACTION,
            random_state=RANDOM_STATE
        )

        sample_parts.append(sampled)

fare_data = pd.concat(
    sample_parts,
    ignore_index=True
)

print("\nSample shape:", fare_data.shape)

print("\nDate coverage:")
print(
    fare_data["pickup_timestamp"].min(),
    "to",
    fare_data["pickup_timestamp"].max()
)

print("\nTarget statistics:")
print(fare_data["base_fare"].describe())

Taxi files found: 12
Sampling: Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv


C:\Users\arudk\AppData\Local\Temp\ipykernel_1532\3339700870.py:46: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(


Sampling: Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv


C:\Users\arudk\AppData\Local\Temp\ipykernel_1532\3339700870.py:46: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(


Sampling: Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv


...

Sampling: Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv


C:\Users\arudk\AppData\Local\Temp\ipykernel_1532\3339700870.py:46: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(


Sampling: Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv


C:\Users\arudk\AppData\Local\Temp\ipykernel_1532\3339700870.py:46: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(



Sample shape: (972035, 9)

Date coverage:
2025-04-01 00:00:31 to 2026-03-31 23:59:49

Target statistics:
count    972035.000000
mean         19.311348
std          19.542125
min       -1047.400000
25%           9.300000
50%          14.200000
75%          24.060000
max         998.000000
Name: base_fare, dtype: float64


## 2. Feature Engineering and Target Quality

The raw pickup timestamp is converted into calendar features that are available at prediction time.

The modelling target is restricted to positive base fares. Negative fares were identified during the dataset quality assessment and are retained in the raw dataset for traceability, but excluded from supervised learning because they do not represent a valid positive fare outcome.

Extreme positive fares are also excluded using a conservative domain-oriented threshold. This prevents a small number of anomalous observations from dominating the regression objective while preserving the majority of legitimate trips.

Categorical identifiers are converted to numeric values so that the tree-based model can use them directly.

In [6]:
# ============================================================
# 2. Feature engineering and target filtering
# ============================================================

# Parse pickup time once and derive features from it.
fare_data["pickup_timestamp"] = pd.to_datetime(
    fare_data["pickup_timestamp"],
    errors="coerce"
)

fare_data["pickup_hour"] = fare_data["pickup_timestamp"].dt.hour
fare_data["pickup_day_of_week"] = fare_data["pickup_timestamp"].dt.dayofweek
fare_data["pickup_day_of_month"] = fare_data["pickup_timestamp"].dt.day
fare_data["pickup_month"] = fare_data["pickup_timestamp"].dt.month
fare_data["is_weekend"] = (
    fare_data["pickup_day_of_week"] >= 5
).astype(int)

# Keep a record of the raw target quality before filtering.
raw_rows = len(fare_data)

negative_fare_count = (
    fare_data["base_fare"] < 0
).sum()

zero_fare_count = (
    fare_data["base_fare"] == 0
).sum()

# ------------------------------------------------------------
# Target quality filtering
# ------------------------------------------------------------

# Negative fares are invalid for supervised fare prediction.
fare_data = fare_data[
    fare_data["base_fare"] > 0
].copy()

# A very high fare can be a legitimate long/expensive trip,
# so use a deliberately conservative upper bound rather than
# an aggressive statistical filter.
fare_data = fare_data[
    fare_data["base_fare"] <= 500
].copy()

# ------------------------------------------------------------
# Basic predictor cleanup
# ------------------------------------------------------------

# Missing numeric predictors are replaced with the median.
# This keeps the modelling pipeline simple and robust.
numeric_features = [
    "rider_count",
    "distance_miles",
    "rate_class_id",
    "origin_loc_id",
    "dest_loc_id"
]

for column in numeric_features:
    fare_data[column] = pd.to_numeric(
        fare_data[column],
        errors="coerce"
    )

    fare_data[column] = fare_data[column].fillna(
        fare_data[column].median()
    )

# Provider and offline flags are treated as categorical codes.
fare_data["provider_code"] = pd.to_numeric(
    fare_data["provider_code"],
    errors="coerce"
).fillna(-1)

fare_data["offline_record_flag"] = (
    fare_data["offline_record_flag"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Convert the mixed offline flag into a stable numeric representation.
fare_data["offline_record_flag"] = (
    fare_data["offline_record_flag"]
    .map({
        "Y": 1,
        "YES": 1,
        "TRUE": 1,
        "1": 1,
        "N": 0,
        "NO": 0,
        "FALSE": 0,
        "0": 0
    })
    .fillna(-1)
)

# Remove records without a usable timestamp.
fare_data = fare_data.dropna(
    subset=["pickup_timestamp"]
).copy()

print("Original sampled rows:", raw_rows)
print("Negative base fares:", negative_fare_count)
print("Zero base fares:", zero_fare_count)

print("\nRows remaining for modelling:", len(fare_data))

print("\nFiltered target statistics:")
print(fare_data["base_fare"].describe())

Original sampled rows: 972035
Negative base fares: 47962
Zero base fares: 509

Rows remaining for modelling: 923552

Filtered target statistics:
count    923552.000000
mean         20.849323
std          18.108099
min           0.010000
25%          10.000000
50%          14.900000
75%          25.020000
max         500.000000
Name: base_fare, dtype: float64


## 3. Leakage-Safe Feature Matrix and Temporal Split

The fare model uses trip and pickup characteristics that can be associated with the fare outcome without directly using the final settlement amounts.

The following fields are intentionally excluded from the feature set:

- `charge_total`
- `toll_total`
- `driver_tip_payment`
- Other monetary settlement components

These variables are either direct components of the final transaction or are only known after the trip, and therefore could leak information about the target.

The dataset is split chronologically rather than randomly. This better reflects the real deployment scenario, where the model is trained on historical trips and applied to future trips.

In [10]:
# ============================================================
# 3. Build the modelling matrix and chronological split
# ============================================================

# Features used for base-fare prediction.
#
# The financial settlement fields are deliberately absent.
# Pickup-time features are derived from information available
# at the start of the trip.
fare_features = [
    "provider_code",
    "rider_count",
    "distance_miles",
    "rate_class_id",
    "offline_record_flag",
    "origin_loc_id",
    "dest_loc_id",
    "pickup_hour",
    "pickup_day_of_week",
    "pickup_day_of_month",
    "pickup_month",
    "is_weekend"
]

target = "base_fare"

# Keep only the columns required by the model.
fare_model_data = fare_data[
    ["pickup_timestamp"] + fare_features + [target]
].copy()

# Ensure chronological ordering before splitting.
fare_model_data = (
    fare_model_data
    .sort_values("pickup_timestamp")
    .reset_index(drop=True)
)

X = fare_model_data[fare_features].copy()
y = fare_model_data[target].copy()

# ------------------------------------------------------------
# Chronological 70 / 15 / 15 split
# ------------------------------------------------------------

n = len(fare_model_data)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train = X.iloc[:train_end]
y_train = y.iloc[:train_end]

X_val = X.iloc[train_end:val_end]
y_val = y.iloc[train_end:val_end]

X_test = X.iloc[val_end:]
y_test = y.iloc[val_end:]

print("Training period:")
print(
    fare_model_data.iloc[0]["pickup_timestamp"],
    "to",
    fare_model_data.iloc[train_end - 1]["pickup_timestamp"]
)
print("Rows:", len(X_train))

print("\nValidation period:")
print(
    fare_model_data.iloc[train_end]["pickup_timestamp"],
    "to",
    fare_model_data.iloc[val_end - 1]["pickup_timestamp"]
)
print("Rows:", len(X_val))

print("\nTest period:")
print(
    fare_model_data.iloc[val_end]["pickup_timestamp"],
    "to",
    fare_model_data.iloc[-1]["pickup_timestamp"]
)
print("Rows:", len(X_test))

print("\nFeature count:", len(fare_features))
print("Features:")
print(fare_features)

Training period:
2025-04-01 00:00:31 to 2025-12-10 09:04:45
Rows: 646486

Validation period:
2025-12-10 09:05:00 to 2026-02-04 08:07:40
Rows: 138533

Test period:
2026-02-04 08:07:50 to 2026-03-31 23:59:49
Rows: 138533

Feature count: 12
Features:
['provider_code', 'rider_count', 'distance_miles', 'rate_class_id', 'offline_record_flag', 'origin_loc_id', 'dest_loc_id', 'pickup_hour', 'pickup_day_of_week', 'pickup_day_of_month', 'pickup_month', 'is_weekend']


## 4. Model Training and Hyperparameter Selection

A tree-based gradient boosting regressor is used because the fare relationship is nonlinear and depends on interactions between trip characteristics, location, time, provider, and rate class.

Two configurations are evaluated on the validation set. The better configuration is selected using validation RMSE, while the held-out test set is used only for the final performance estimate.

This approach provides a lightweight form of hyperparameter tuning that is practical for the approximately 924,000-row modelling sample.## 4. Model Training and Hyperparameter Selection

A tree-based gradient boosting regressor is used because the fare relationship is nonlinear and depends on interactions between trip characteristics, location, time, provider, and rate class.

Two configurations are evaluated on the validation set. The better configuration is selected using validation RMSE, while the held-out test set is used only for the final performance estimate.

This approach provides a lightweight form of hyperparameter tuning that is practical for the approximately 924,000-row modelling sample.

In [13]:
# ============================================================
# 4. Train and compare gradient-boosting configurations
# ============================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# LightGBM is preferred for this tabular dataset because it is
# efficient on large datasets and captures nonlinear feature
# interactions well.
try:
    from lightgbm import LGBMRegressor

    use_lightgbm = True
    print("LightGBM is available. Using LightGBM.")

except ImportError:
    from sklearn.ensemble import HistGradientBoostingRegressor

    use_lightgbm = False
    print("LightGBM is unavailable. Using HistGradientBoostingRegressor.")


# ------------------------------------------------------------
# Candidate configurations
# ------------------------------------------------------------

if use_lightgbm:

    candidate_models = {
        "LightGBM_Baseline": LGBMRegressor(
            n_estimators=500,
            learning_rate=0.05,
            num_leaves=31,
            max_depth=-1,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            verbosity=-1
        ),

        "LightGBM_Tuned": LGBMRegressor(
            n_estimators=800,
            learning_rate=0.03,
            num_leaves=63,
            max_depth=-1,
            min_child_samples=30,
            subsample=0.85,
            colsample_bytree=0.9,
            reg_alpha=0.1,
            reg_lambda=0.5,
            random_state=42,
            verbosity=-1
        )
    }

else:

    candidate_models = {
        "HistGradientBoosting_Baseline": HistGradientBoostingRegressor(
            max_iter=300,
            learning_rate=0.05,
            max_leaf_nodes=31,
            l2_regularization=1.0,
            random_state=42
        ),

        "HistGradientBoosting_Tuned": HistGradientBoostingRegressor(
            max_iter=500,
            learning_rate=0.03,
            max_leaf_nodes=63,
            l2_regularization=2.0,
            random_state=42
        )
    }


# ------------------------------------------------------------
# Validation-based model selection
# ------------------------------------------------------------

tuning_results = []
trained_candidates = {}

for model_name, candidate in candidate_models.items():

    print(f"\nTraining: {model_name}")

    candidate.fit(X_train, y_train)

    val_prediction = candidate.predict(X_val)

    # Fare cannot be negative.
    val_prediction = np.maximum(val_prediction, 0)

    val_mae = mean_absolute_error(
        y_val,
        val_prediction
    )

    val_rmse = np.sqrt(
        mean_squared_error(
            y_val,
            val_prediction
        )
    )

    val_r2 = r2_score(
        y_val,
        val_prediction
    )

    tuning_results.append({
        "model": model_name,
        "validation_mae": val_mae,
        "validation_rmse": val_rmse,
        "validation_r2": val_r2
    })

    trained_candidates[model_name] = candidate

    print(f"Validation MAE : {val_mae:.4f}")
    print(f"Validation RMSE: {val_rmse:.4f}")
    print(f"Validation R²  : {val_r2:.4f}")


# ------------------------------------------------------------
# Select the configuration with the lowest validation RMSE
# ------------------------------------------------------------

tuning_results = pd.DataFrame(tuning_results)

best_model_name = (
    tuning_results
    .sort_values("validation_rmse")
    .iloc[0]["model"]
)

fare_model = trained_candidates[best_model_name]

print("\nSelected model:", best_model_name)

print("\nValidation comparison:")
display(
    tuning_results.sort_values("validation_rmse")
)

LightGBM is unavailable. Using HistGradientBoostingRegressor.

Training: HistGradientBoosting_Baseline


C:\Users\arudk\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\arudk\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\arudk\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\arudk\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\arudk\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreatePro

Validation MAE : 4.2448
Validation RMSE: 8.4779
Validation R²  : 0.7839

Training: HistGradientBoosting_Tuned
Validation MAE : 4.1746
Validation RMSE: 8.3951
Validation R²  : 0.7881

Selected model: HistGradientBoosting_Tuned

Validation comparison:


,model,validation_mae,validation_rmse,validation_r2
1,HistGradientBoosting_Tuned,4.174575,8.395087,0.788093
0,HistGradientBoosting_Baseline,4.244843,8.477882,0.783893


## 5. Final Test Evaluation

The best-performing configuration is evaluated on the held-out test period.

The test set has not been used for model selection or hyperparameter tuning, so these metrics provide the final estimate of how the fare prediction model performs on unseen future observations.

MAE measures the average absolute fare prediction error, RMSE penalizes larger errors more strongly, and R² measures the proportion of target variance explained by the model.

In [18]:
# ============================================================
# Final evaluation on the held-out test period
# ============================================================

# Generate predictions only after selecting the best model.
test_prediction = fare_model.predict(X_test)

# A fare prediction cannot be negative.
test_prediction = np.maximum(test_prediction, 0)

# Calculate the three competition metrics.
fare_test_mae = mean_absolute_error(
    y_test,
    test_prediction
)

fare_test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        test_prediction
    )
)

fare_test_r2 = r2_score(
    y_test,
    test_prediction
)

print("Selected model:", best_model_name)

print("\nFinal Test Performance")
print(f"MAE : {fare_test_mae:.4f}")
print(f"RMSE: {fare_test_rmse:.4f}")
print(f"R²  : {fare_test_r2:.4f}")

# A small prediction sample makes it easy to sanity-check
# the relationship between actual and predicted fares.
comparison = pd.DataFrame({
    "actual_base_fare": y_test.iloc[:15].values,
    "predicted_base_fare": test_prediction[:15]
})

print("\nSample predictions:")
display(comparison)

Selected model: HistGradientBoosting_Tuned

Final Test Performance
MAE : 3.9314
RMSE: 8.1008
R²  : 0.7944

Sample predictions:


,actual_base_fare,predicted_base_fare
0,73.70,49.282371
1,7.20,6.337249
2,53.40,50.688173
3,7.20,7.146712
4,23.17,15.874559
5,5.10,6.362893
6,36.20,12.909468
7,14.20,13.048420
8,13.50,12.614439
9,17.00,15.288237


## 6. Model Artifact and Performance Record

The selected HistGradientBoosting model is saved as the final fare prediction artifact.

The validation comparison and held-out test performance are also exported so that the results can be reproduced and incorporated directly into the final technical report.

In [23]:
# joblib is used to serialize the trained model to a .pkl artifact.
import joblib

print("joblib imported successfully.")

joblib imported successfully.


In [25]:
# ============================================================
# Save the final fare model and evaluation results
# ============================================================

MODEL_DIR = BASE_DIR / "models" / "fare"
OUTPUT_DIR = BASE_DIR / "outputs"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save the trained model.
fare_model_path = MODEL_DIR / "base_fare_model.pkl"
joblib.dump(fare_model, fare_model_path)

# Combine model-selection and final test results into one
# report-ready record.
fare_results = tuning_results.copy()

fare_results["test_mae"] = np.nan
fare_results["test_rmse"] = np.nan
fare_results["test_r2"] = np.nan

selected_mask = fare_results["model"] == best_model_name

fare_results.loc[selected_mask, "test_mae"] = fare_test_mae
fare_results.loc[selected_mask, "test_rmse"] = fare_test_rmse
fare_results.loc[selected_mask, "test_r2"] = fare_test_r2

fare_results_path = OUTPUT_DIR / "fare_model_results.csv"
fare_results.to_csv(fare_results_path, index=False)

print("Fare model saved to:")
print(fare_model_path)

print("\nFare model results saved to:")
print(fare_results_path)

print("\nFinal fare-model results:")
display(fare_results)

Fare model saved to:
C:\Users\arudk\Downloads\UrbanFlow_AI\models\fare\base_fare_model.pkl

Fare model results saved to:
C:\Users\arudk\Downloads\UrbanFlow_AI\outputs\fare_model_results.csv

Final fare-model results:


,model,validation_mae,validation_rmse,validation_r2,test_mae,test_rmse,test_r2
0,HistGradientBoosting_Baseline,4.244843,8.477882,0.783893,NaN,NaN,NaN
1,HistGradientBoosting_Tuned,4.174575,8.395087,0.788093,3.931388,8.100816,0.794363
